In [ ]:
from pydrake.geometry import StartMeshcat
meshcat = StartMeshcat()

In [ ]:
# Load spinner problem
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy import sparse

plt.style.use('bmh')
plt.rcParams.update({'font.size': 20})

from pydrake.systems.drawing import plot_system_graphviz
from pydrake.solvers import MathematicalProgram, CommonSolverOption, SolverOptions, SnoptSolver
from pydrake.systems.drawing import plot_system_graphviz
from pydrake.geometry import Rgba
from pydrake.symbolic import Evaluate
from pydrake.solvers import Solve, ClarabelSolver

# Import utils for setting up and visualizing problem
import sys, os, importlib
sys.path.append(os.path.join(os.getcwd(), "../src"))
import system_utils, vis_utils, to_utils, contact_models
importlib.reload(system_utils)
importlib.reload(vis_utils)
importlib.reload(to_utils)
importlib.reload(contact_models)
from system_utils import create_scene, create_sim
from vis_utils import animate_with_vecs, animate
from to_utils import KnotPoint, KnotPointCache, ContactCache, CITOProb
from contact_models import NoSlipForceModel, MaxDis_NormalCompForceModel, retraction_map, inv_retraction_map
from pydrake.autodiffutils import AutoDiffXd, ExtractGradient, ExtractValue
from pydrake.symbolic import MakeVectorContinuousVariable

from pydrake.geometry import SceneGraph
from pydrake.multibody.parsing import Parser
from pydrake.multibody.plant import AddMultibodyPlantSceneGraph
from pydrake.systems.analysis import Simulator
from pydrake.systems.framework import DiagramBuilder
from pydrake.visualization import AddDefaultVisualization, AddFrameTriadIllustration

# Create a scene consisting of a plant (loaded from a URDF) and scene_graph
# Optionally adds default visualization if add_vis = True, and a meshcat instance is passed in
def create_scene(urdf, time_step, add_vis = False, meshcat=None, add_elem_func=None, show_frames = False):
    # Set up builder for diagram
    builder = DiagramBuilder()
    scene_graph = SceneGraph()
    scene_graph.set_name("scene_graph")
    # plant = builder.AddNamedSystem("plant", MultibodyPlant(time_step=sim_time_step))
    plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=time_step, scene_graph=scene_graph)

    # plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step = sim_time_step)
    parser = Parser(builder)

    # Load in the model
    if urdf is not None:
        parser.AddModels(urdf)
    if not add_elem_func is None:
        add_elem_func(plant, parser) 
    plant.Finalize()

    # Add visualization
    if add_vis:
        # Clean up Meshcat instance
        meshcat.Delete()
        meshcat.DeleteAddedControls()

        if show_frames:
            inspector = scene_graph.model_inspector()
            for frame_id in inspector.GetAllFrameIds():
                AddFrameTriadIllustration(plant=plant, scene_graph=scene_graph,frame_id=frame_id)
        AddDefaultVisualization(builder=builder, meshcat=meshcat)

    diagram = builder.Build()
    return diagram

# Creates a diagram with a plant (loaded from URDF), scene_graph and default visualizer
# and a corresponding simulator (this is mainly used to visualize things and do meshcat recordings)
def create_sim(meshcat, urdf, time_step, add_elem_func=None, show_frames = False):
    diagram = create_scene(urdf, time_step, add_vis = True, meshcat=meshcat, add_elem_func=add_elem_func, show_frames=show_frames)
    context = diagram.CreateDefaultContext()
    sim = Simulator(diagram, context)
    sim.Initialize()
    return sim, diagram